# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vikraamkumar-ds/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Two paper findings + my methodology questions

Source: *FlyRank — The State of AI-Driven SEO*, March 2026, ML Appendix.

### Finding 1 — “What Predicts Growth?” (logistic regression, 71% holdout accuracy, 80/20 split)

The paper trains a logistic regression on `content_age`, `days_since_update`, `days_visible`, `avg_position`, `word_count`, and others to separate growing pages from declining ones, reporting 71% holdout accuracy on an 80/20 split.

- **Where does the label come from?** The paper's own “Trend Direction” definition: growth/decline is “calculated from 30d-vs-prev-30d impression change” (up >10%, down >10%). Some of the model's own features (e.g. `days_visible`, tied to the 90-day activity window) plausibly overlap that same recent period the label is measuring — the same overlap risk I found and documented for my own `visible_queries`/`rare_share` features in `w03_feature_leakage_check`.
- **Does the validation design carry the claim?** The paper states an 80/20 split but does not say whether it is grouped by brand. The dataset spans only **57 brands** across 341,701 pages — far fewer brands than pages. If the split is a plain random row split, pages from the same brand almost certainly appear in both the train and test sets, which is exactly the leakage risk my own `w05`/`w06` grouped-split work was built to rule out. A 71% accuracy figure from a brand-mixed split and a 71% figure from a brand-held-out split would support very different claims (“this works on content patterns in general” vs. “this works on brands we've already seen”).
- **My question, stated constructively:** *Was the 80/20 split for the growth model grouped by brand, or a random row split across all 57 brands? If it's row-level, would you be open to re-running it as a brand-grouped split to see whether the 71% holds — the same before/after check I ran on my own model in Section 2 below?*

### Finding 2 — “What Predicts Health?” (Random Forest, Average Position = 43% importance)

The paper reports Average Position as the top predictor of Health Score (43% importance), ahead of Impressions (32%) and Scroll Depth (15%) — and is admirably upfront that this is descriptive, not causal, since Health Score itself is a composite built from **Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts)**.

- **Where does the label come from?** Directly from a formula that includes Position as one of its four ingredients (30 of 100 points). Position isn't just correlated with the label — it's a literal component of how the label is computed.
- **Does the validation design carry the claim?** A holdout split protects against overfitting to noise, but it cannot rule out this specific issue, because the same relationship exists identically in both the training and the held-out fold — no split design can separate a feature from a label it partially defines. The paper already flags this in one sentence (“importance is descriptive rather than causal”), which is the right call, but doesn't go the extra step of quantifying it.
- **My question, stated constructively:** *Since Position and Impressions are literally inside the Health Score formula, would re-running the same Random Forest with those two components excluded — keeping only the non-constituent features like Content Age, Word Count, and Days Visible — show whether any of them carry real independent signal once the mechanical overlap is removed? That would turn “descriptive” into a number a reader could act on.*

*(Tone check: both questions above ask for a specific, runnable next step — not a takedown of the paper's conclusions, which are honestly framed and already self-aware about several of their own limits.)*


In [1]:
# Nothing to compute here yet — fill in the markdown cell above with the two real findings first.
print('Section 1 is a written methodology critique — see markdown cell above.')


Section 1 is a written methodology critique — see markdown cell above.


## 2. My model under an honest split (before/after)

**“After” is already in `w05_model.ipynb`** — that notebook used `GroupShuffleSplit` on `client_hash_id` from the start (F1 = 0.537 on Random Forest, per that notebook's real output).

**“Before”, built below:** the same model, same features, same hyperparameters — but a plain random row split with no grouping. This is the split someone would use by default if they hadn't stopped to ask whether rows from the same client leak into both train and test.

**What to watch for:** if the random-split numbers look meaningfully *better* than the grouped ones, that gap is the honest cost of the earlier mistake — the random split was letting the model partly recognize *which client* a row belongs to, not learn a pattern that generalizes to unseen clients.


In [2]:
%pip -q install duckdb huggingface_hub pandas scikit-learn matplotlib

import os, getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily':     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

features = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}),
    windowed AS (
        SELECT
            f.client_hash_id, f.content_hash_id,
            SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_prev30,
            AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_prev30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()
features['ctr_prev30'] = features['clk_prev30'] / features['imp_prev30']

qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)  AS visible_queries,
           ANY_VALUE(rare_impressions_share)        AS rare_share,
           ANY_VALUE(anonymized_impressions_share)  AS anon_share,
           MAX(impressions_90d)                     AS top_query_impressions,
           SUM(impressions_90d)                     AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()
qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']

data = features.merge(qsignals, on='content_hash_id', how='left')
data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

FEATURE_COLS = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'ctr_prev30',
                'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=FEATURE_COLS + ['is_declining']).reset_index(drop=True)
X = model_data[FEATURE_COLS]
y = model_data['is_declining']
groups = model_data['client_hash_id']
print(f'{len(model_data):,} rows, base rate = {y.mean():.3f}')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

102,203 rows, base rate = 0.633


In [3]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

def fit_and_score(X_tr, X_te, y_tr, y_te, label):
    m = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                                random_state=42, n_jobs=-1, class_weight='balanced')
    m.fit(X_tr, y_tr)
    pred = m.predict(X_te)
    return {
        'split': label,
        'precision': precision_score(y_te, pred, zero_division=0),
        'recall':    recall_score(y_te, pred, zero_division=0),
        'f1':        f1_score(y_te, pred, zero_division=0),
    }

# BEFORE: naive random split, no grouping — same test_size, same seed, for fair comparison
X_tr_rand, X_te_rand, y_tr_rand, y_te_rand = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
before = fit_and_score(X_tr_rand, X_te_rand, y_tr_rand, y_te_rand, 'BEFORE — random row split (dishonest)')

# AFTER: grouped by client — same approach already used in w05_model.ipynb
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_tr_grp, X_te_grp = X.iloc[train_idx], X.iloc[test_idx]
y_tr_grp, y_te_grp = y.iloc[train_idx], y.iloc[test_idx]
after = fit_and_score(X_tr_grp, X_te_grp, y_tr_grp, y_te_grp, 'AFTER — grouped by client (honest)')

before_after = pd.DataFrame([before, after])
print(before_after.to_string(index=False))
print()
gap = before['f1'] - after['f1']
print(f'F1 gap (before - after): {gap:+.3f}')
print('Positive gap = the random split was overstating performance; that gap is the honest cost')
print('of not grouping by client, not a real capability the model has.')


                                split  precision   recall       f1
BEFORE — random row split (dishonest)   0.749317 0.645093 0.693310
   AFTER — grouped by client (honest)   0.780663 0.409761 0.537431

F1 gap (before - after): +0.156
Positive gap = the random split was overstating performance; that gap is the honest cost
of not grouping by client, not a real capability the model has.


## 3. Leakage audit

Same three-part hunt from `w03_feature_leakage_check.ipynb`, re-run here against the **final feature set** used for the actual w05 model, on the honest (grouped) split's training data.


In [4]:
from sklearn.metrics import roc_auc_score

# Test 1: label-defining column must not be a feature
assert 'imp_last30' not in FEATURE_COLS, 'LEAKAGE: outcome column found inside FEATURE_COLS'
print('PASS -- imp_last30 (used to build the label) is excluded from FEATURE_COLS')

# Test 2: single-feature AUC sweep, run on the honest training fold only
print(f"\n{'feature':<18}{'single-feature AUC':>20}")
for col in FEATURE_COLS:
    auc = roc_auc_score(y_tr_grp, X_tr_grp[[col]].rank(pct=True).iloc[:, 0])
    flag = '  <-- investigate' if auc > 0.90 else ''
    print(f'{col:<18}{auc:>20.3f}{flag}')

# Test 3: confirm the query-mix window-overlap caveat still holds for this final feature set
panel_bounds = con.sql(f"SELECT MAX(report_date) AS max_d FROM {TABLES['fact_daily']}").df()
end_d = panel_bounds['max_d'].iloc[0]
outcome_start = end_d - pd.Timedelta(days=30)
query_window_start = end_d - pd.Timedelta(days=90)
print(f"\nOutcome window starts: {outcome_start}   Query-mix (90d) window starts: {query_window_start}")
print('Still overlapping:', query_window_start < outcome_start,
      '-- unchanged from w03: query-mix features (visible_queries, rare_share, anon_share, top_query_share)')
print('remain directional/descriptive, not strictly pre-outcome. Carried into Section 4 as a claim limit.')


PASS -- imp_last30 (used to build the label) is excluded from FEATURE_COLS

feature             single-feature AUC
imp_prev30                       0.515
clk_prev30                       0.443
pos_prev30                       0.616
ctr_prev30                       0.404
visible_queries                  0.426
rare_share                       0.455
anon_share                       0.546
top_query_share                  0.555


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Outcome window starts: 2026-05-31 00:00:00   Query-mix (90d) window starts: 2026-04-01 00:00:00
Still overlapping: True -- unchanged from w03: query-mix features (visible_queries, rare_share, anon_share, top_query_share)
remain directional/descriptive, not strictly pre-outcome. Carried into Section 4 as a claim limit.


## 4. Claim rewrite

**Boldest sentence, as originally written (from `w05_model.ipynb`, Section 1):**
> “Random Forest classifier... predicting `is_declining`” — read plainly, this implies the model *knows* which pages will decline.

**Rewritten in safe language:**
> On a held-out group of clients the model never trained on, the Random Forest's decline flag was **observed** to align with actual 30-day impression drops at a precision of 0.78 and recall of 0.41 (per w05's grouped-split results) — meaning it correctly flags roughly 4 in 10 truly declining pages, and about 78% of its flags are right when it does flag one. This is a **directional, decision-support** signal for triage, not a guarantee about any individual page, and the query-mix features it partly relies on have a known **window-overlap limitation** (Section 3) that keeps this claim short of a strictly causal or fully pre-outcome prediction.


In [5]:
# No computation needed for the rewrite itself — this documents the before/after language.
# Optional: pull the real recall/precision numbers programmatically so the sentence above self-updates.
print(f"Grouped-split (honest) result this rewrite is based on: precision={after['precision']:.3f}, "
      f"recall={after['recall']:.3f}, f1={after['f1']:.3f}")


Grouped-split (honest) result this rewrite is based on: precision=0.781, recall=0.410, f1=0.537


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
